# 데이터 테이블 규칙 시험 (test_data_rules)
잘못된 입력 7가지를 일부러 넣어 보고, SQLite가 모두 거부하는지 확인합니다.
시험이 끝날 때마다 `rollback`으로 취소하므로 **DB는 바뀌지 않습니다.**

- **준비:** 같은 폴더에 `patent_final.db`가 있어야 합니다. 없으면 `load_final.ipynb`를 먼저 실행하세요.

## 1. DB 연결

In [2]:
import sqlite3, os
DB = 'patent_final.db'
assert os.path.exists(DB), 'patent_final.db가 없습니다. load_final.ipynb를 먼저 실행하세요.'
con = sqlite3.connect(DB)
con.execute('PRAGMA foreign_keys = ON')       # 외래키 검사 켜기
print('연결 완료:', DB, '/ SQLite', sqlite3.sqlite_version)

연결 완료: patent_final.db / SQLite 3.50.4


## 2. 잘못된 입력 7가지
| # | 잘못된 입력 | 막는 규칙 |
|---|---|---|
| 1 | 등록번호만 있고 등록일 없음 | CHECK (둘 다 있거나 둘 다 NULL) |
| 2 | 출원일보다 이른 공개일 | CHECK (출원일 ≤ 공개일) |
| 3 | 허용되지 않은 법적상태('심사중') | CHECK (허용값 6개) |
| 4 | 피인용 음수(-1) | CHECK (0 이상) |
| 5 | 출원번호 중복 | UNIQUE |
| 6 | 없는 특허에 출원인 연결 | 외래키(FK) |
| 7 | 같은 특허에 같은 IPC 두 번 | 기본키(PK) |

In [3]:
tests = [
 ("등록번호만 있고 등록일 없음", "INSERT INTO patent(application_no,title_ko,application_date,legal_status,abstract,registration_no) VALUES ('1029999999999','t','2015-01-01','등록','a','1099999990000')"),
 ("출원일보다 이른 공개일", "INSERT INTO patent(application_no,title_ko,application_date,legal_status,abstract,publication_date) VALUES ('1029999999998','t','2015-05-01','공개','a','2015-01-01')"),
 ("허용되지 않은 법적상태", "INSERT INTO patent(application_no,title_ko,application_date,legal_status,abstract) VALUES ('1029999999997','t','2015-01-01','심사중','a')"),
 ("피인용 음수", "INSERT INTO patent(application_no,title_ko,application_date,legal_status,abstract,cited_count) VALUES ('1029999999996','t','2015-01-01','공개','a',-1)"),
 ("출원번호 중복", "INSERT INTO patent(application_no,title_ko,application_date,legal_status,abstract) SELECT application_no,'t','2015-01-01','공개','a' FROM patent LIMIT 1"),
 ("없는 특허에 출원인 연결", "INSERT INTO patent_applicant VALUES (999999,1,1)"),
 ("같은 특허에 같은 IPC 두 번", "INSERT INTO patent_ipc SELECT patent_id,ipc_code,ipc_subclass,99 FROM patent_ipc LIMIT 1"),
]

## 3. 시험 실행
하나씩 넣어 보고, 거부되면 PASS · 들어가 버리면 FAIL로 표시합니다.

In [4]:
ok = 0
for name, sql in tests:
    try:
        con.execute(sql)
        print('FAIL(통과됨):', name)
    except sqlite3.IntegrityError as e:
        ok += 1
        print('PASS(거부):', name, '-', e)
    con.rollback()                              # 시험한 입력 취소 (DB 그대로)
print(f'{ok}/{len(tests)} 규칙 정상')

PASS(거부): 등록번호만 있고 등록일 없음 - CHECK constraint failed: (registration_no IS NULL) = (registration_date IS NULL)
PASS(거부): 출원일보다 이른 공개일 - CHECK constraint failed: publication_date  IS NULL OR application_date <= publication_date
PASS(거부): 허용되지 않은 법적상태 - CHECK constraint failed: legal_status IN ('공개','등록','소멸','취하','거절','포기')
PASS(거부): 피인용 음수 - CHECK constraint failed: cited_count IS NULL OR cited_count >= 0
PASS(거부): 출원번호 중복 - UNIQUE constraint failed: patent.application_no
PASS(거부): 없는 특허에 출원인 연결 - FOREIGN KEY constraint failed
PASS(거부): 같은 특허에 같은 IPC 두 번 - UNIQUE constraint failed: patent_ipc.patent_id, patent_ipc.ipc_code
7/7 규칙 정상


## 4. DB가 바뀌지 않았는지 확인

In [5]:
for t in ['patent', 'applicant', 'patent_applicant', 'patent_ipc']:
    print(t, con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0])
con.close()

patent 3122
applicant 1246
patent_applicant 3400
patent_ipc 15250
